In [1]:
from tree import Mbtree
from marubatsu import Marubatsu

def calc_playout_prob_by_df(self, node):
    node.playout_prob = {
        node.mb.CIRCLE: 0.0,
        node.mb.CROSS: 0.0,
        node.mb.DRAW: 0.0
    }
    if node.mb.status != Marubatsu.PLAYING:
        node.playout_prob[node.mb.status] = 1.0
    else:
        childnum = len(node.children)
        for childnode in node.children:
            self.calc_playout_prob_by_df(childnode)
            for status, prob in childnode.playout_prob.items():
                node.playout_prob[status] += prob / childnum
                 
Mbtree.calc_playout_prob_by_df = calc_playout_prob_by_df

In [2]:
from tree import Node

def __init__(self, algo="bf", shortest_victory=False, 
             recalculate_draw_score=False, subtree=None):
    if subtree is not None:
        self.subtree = subtree
        self.create_subtree()
        return

    self.algo = algo
    self.shortest_victory = shortest_victory
    self.recalculate_draw_score = recalculate_draw_score
    Node.count = 0
    self.root = Node(Marubatsu())
    if self.algo == "bf":  
        self.create_tree_by_bf()
        self.calc_score_by_bf()
    else:
        self.nodelist = [self.root]
        self.nodelist_by_depth = [[] for _ in range(10)]
        self.nodelist_by_depth[0].append(self.root)
        self.nodenum = 0
        self.create_tree_by_df(self.root)
        self.nodelist_by_score = []
        self.calc_score_by_df(self.root)
    self.calc_playout_prob_by_df(self.root)
    if self.recalculate_draw_score:
        self.recalculate_score()
    self.nodelist_by_mb = {}
    self.bestmoves_by_mb = {}
    self.calc_bestmoves(self.root)
    
Mbtree.__init__ = __init__

In [3]:
mbtree = Mbtree()
for node in [mbtree.root] + mbtree.root.children:
    print(node.mb)
    print(f"o    {node.playout_prob[node.mb.CIRCLE] * 100:5.1f}%")
    print(f"x    {node.playout_prob[node.mb.CROSS] * 100:5.1f}%")
    print(f"draw {node.playout_prob[node.mb.DRAW] * 100:5.1f}%")
    print()

     9 depth 1 node created
    72 depth 2 node created
   504 depth 3 node created
  3024 depth 4 node created
 15120 depth 5 node created
 54720 depth 6 node created
148176 depth 7 node created
200448 depth 8 node created
127872 depth 9 node created
     0 depth 10 node created
total node num = 549946
Turn o
...
...
...

o     58.5%
x     28.8%
draw  12.7%

Turn x
O..
...
...

o     60.7%
x     26.4%
draw  12.9%

Turn x
...
O..
...

o     53.6%
x     33.6%
draw  12.9%

Turn x
...
...
O..

o     60.7%
x     26.4%
draw  12.9%

Turn x
.O.
...
...

o     53.6%
x     33.6%
draw  12.9%

Turn x
...
.O.
...

o     69.3%
x     19.3%
draw  11.4%

Turn x
...
...
.O.

o     53.6%
x     33.6%
draw  12.9%

Turn x
..O
...
...

o     60.7%
x     26.4%
draw  12.9%

Turn x
...
..O
...

o     53.6%
x     33.6%
draw  12.9%

Turn x
...
...
..O

o     60.7%
x     26.4%
draw  12.9%



In [4]:
mbtree.save("../data/playout")

save completed.


In [5]:
mbtree2 = Mbtree.load("../data/playout")
print(mbtree2.root.playout_prob)

{0: 0.5849206349206348, 1: 0.2880952380952381, 'draw': 0.126984126984127}


In [ ]:
from time import perf_counter
from copy import deepcopy
import random 

def playout(mborig, pnum, timelimit=None):
    if timelimit is not None:   
        starttime = perf_counter()
        timelimit_pc = starttime + timelimit        
    result = {}
    for move in mborig.calc_legal_moves():
        result[move] = {
            mborig.CIRCLE: 0,
            mborig.CROSS: 0,
            mborig.DRAW: 0,
        }
    count = 0
    for _ in range(pnum):
        if timelimit is not None and perf_counter() > timelimit_pc:
            break
        mb = deepcopy(mborig)
        firstmove = None
        while mb.status == mb.PLAYING:
            move = random.choice(mb.calc_legal_moves())
            if firstmove is None:
                firstmove = move
            mb.move(move)
        result[firstmove][mb.status] += 1
        count += 1
    return {
        "result": result,
        "count": count,
    }

In [7]:
mb = Marubatsu()
print(mb)
retval = playout(mb, 10000)
print(f"playout count = {retval['count']}")
for move, counts in retval["result"].items():
    print()
    print("---------------------")
    print()
    print(f"move {mb.board.move_to_xy(move)}")
    mb.move(move)
    print(mb)
    mb.unmove()
    total = sum(counts.values())
    for status, count in counts.items():
        print(f"{mb.board.MARK_TABLE[status]:4s}: {count:7d} {count/total*100:6.1f}%")

Turn o
...
...
...

playout count = 10000

---------------------

move (0, 0)
Turn x
O..
...
...

o   :     710   61.0%
x   :     298   25.6%
draw:     156   13.4%

---------------------

move (0, 1)
Turn x
...
O..
...

o   :     574   55.5%
x   :     329   31.8%
draw:     132   12.8%

---------------------

move (0, 2)
Turn x
...
...
O..

o   :     670   60.2%
x   :     307   27.6%
draw:     136   12.2%

---------------------

move (1, 0)
Turn x
.O.
...
...

o   :     596   51.5%
x   :     418   36.1%
draw:     143   12.4%

---------------------

move (1, 1)
Turn x
...
.O.
...

o   :     761   68.4%
x   :     202   18.2%
draw:     149   13.4%

---------------------

move (1, 2)
Turn x
...
...
.O.

o   :     620   54.6%
x   :     365   32.1%
draw:     151   13.3%

---------------------

move (2, 0)
Turn x
..O
...
...

o   :     675   62.6%
x   :     263   24.4%
draw:     141   13.1%

---------------------

move (2, 1)
Turn x
...
..O
...

o   :     593   53.5%
x   :     374   33.7%
draw

In [8]:
retval = playout(mb, 1000000, timelimit=1)
print(f"playout count = {retval['count']}")

playout count = 25758


In [ ]:
def analyze_playout(mb, retval, mbtree):
    print(f"playout count = {retval['count']}")
    diffsum = 0
    diffcount = 0
    for move, counts in retval["result"].items():
        print(f"move {mb.board.move_to_xy(move)}")
        mb.move(move)
        print(mb)
        node = mbtree.nodelist_by_mb[tuple(mb.records)]
        mb.unmove()
        total = sum(counts.values())
        print("        count   ratio    prob    diff")
        for status, count in counts.items():
            mark = mb.board.MARK_TABLE[status]
            ratio = count / total * 100
            prob = node.playout_prob[status] * 100
            diff = abs(ratio - prob)
            diffsum += diff
            diffcount += 1
            print(f"{mark:4s}: {count:7d} {ratio:6.2f}% {prob:6.2f}% {diff:6.2f}%")
        print()
        print("-------------------------------------")
        print()
    print(f"diff average = {diffsum / diffcount:6.2f}%")
    print()

In [ ]:
for pnum in [100, 1000, 10000, 100000, 1000000]:
    retval = playout(mb, pnum=pnum)
    analyze_playout(mb, retval, mbtree)

playout count = 100
move (0, 0)
Turn x
O..
...
...

        count   ratio    prob    diff
o   :      14  73.68%  60.71%  12.97%
x   :       5  26.32%  26.43%   0.11%
draw:       0   0.00%  12.86%  12.86%

-------------------------------------

move (0, 1)
Turn x
...
O..
...

        count   ratio    prob    diff
o   :       2  16.67%  53.57%  36.90%
x   :       6  50.00%  33.57%  16.43%
draw:       4  33.33%  12.86%  20.48%

-------------------------------------

move (0, 2)
Turn x
...
...
O..

        count   ratio    prob    diff
o   :       7  70.00%  60.71%   9.29%
x   :       3  30.00%  26.43%   3.57%
draw:       0   0.00%  12.86%  12.86%

-------------------------------------

move (1, 0)
Turn x
.O.
...
...

        count   ratio    prob    diff
o   :       1  16.67%  53.57%  36.90%
x   :       3  50.00%  33.57%  16.43%
draw:       2  33.33%  12.86%  20.48%

-------------------------------------

move (1, 1)
Turn x
...
.O.
...

        count   ratio    prob    diff
o   :      11 